# Concurso Docente UBA - Extracción Automática desde cv_es.yaml

Este notebook genera texto base para copiar/pegar en el nuevo sistema de concursos docentes.

Uso recomendado:
1. Ejecutar la celda 2 (setup).
2. Ejecutar cada celda de sección (a, b, c, ...).
3. Copiar el resultado y ajustar redacción final si hace falta.

In [ ]:
import yaml
from pathlib import Path
from datetime import date

DATA_PATH = Path("../cv_db/cv_es.yaml")

with DATA_PATH.open("r", encoding="utf-8") as f:
    cv_data = yaml.safe_load(f)

print(f"Datos cargados desde: {DATA_PATH.resolve()}")

def as_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]

def list_to_text(value):
    out = []
    for item in as_list(value):
        if isinstance(item, dict):
            for k, v in item.items():
                out.append(f"{k}: {v}")
        else:
            out.append(str(item))
    return [x.strip() for x in out if str(x).strip()]

def get_path(*keys, default=None):
    node = cv_data
    for key in keys:
        if not isinstance(node, dict) or key not in node:
            return [] if default is None else default
        node = node[key]
    return node

def print_title(title):
    bar = "=" * len(title)
    print(bar)
    print(title)
    print(bar)

def format_date(d):
    return str(d or "").strip()

def is_ongoing(date_str):
    return "actual" in str(date_str).lower()

today = date.today().isoformat()
print(f"Fecha de ejecución: {today}")

## a. TITULOS UNIVERSITARIOS OBTENIDOS

In [ ]:
print_title("a) TÍTULOS UNIVERSITARIOS OBTENIDOS")

titulos = get_path("Educación", "Educación Universitaria")
for i, t in enumerate(as_list(titulos), 1):
    nombre = t.get("name", "")
    institucion = t.get("location", "")
    periodo = format_date(t.get("date", ""))
    detalles = "; ".join(list_to_text(t.get("description")))

    print(f"{i}. {nombre}")
    print(f"   Facultad/Universidad: {institucion}")
    print(f"   Período: {periodo}")
    if detalles:
        print(f"   Detalles: {detalles}")
    print()

## b. ANTECEDENTES DOCENTES E ÍNDOLE DE LAS TAREAS

In [ ]:
print_title("b) ANTECEDENTES DOCENTES E ÍNDOLE DE LAS TAREAS")

docencia = get_path("Experiencia", "Docencia y Formación")

def naturaleza_designacion(nombre):
    low = str(nombre).lower()
    if "invitado" in low:
        return "Invitado"
    if "facilitador" in low:
        return "Facilitador"
    if "co-organizador" in low or "organizador" in low:
        return "Organizador / Docente"
    return "Regular"

for i, item in enumerate(as_list(docencia), 1):
    nombre = item.get("name", "")
    periodo = format_date(item.get("date", ""))
    institucion = item.get("location", "")
    designacion = naturaleza_designacion(nombre)
    tareas = list_to_text(item.get("description"))

    print(f"{i}. {nombre}")
    print(f"   Institución/lugar: {institucion}")
    print(f"   Período: {periodo}")
    print(f"   Naturaleza de la designación: {designacion}")
    print("   Tareas desarrolladas:")
    for t in tareas:
        print(f"   - {t}")
    print()

## c. ANTECEDENTES CIENTÍFICOS Y PUBLICACIONES

In [ ]:
print_title("c) ANTECEDENTES CIENTÍFICOS")

print("PUBLICACIONES:")
publicaciones = get_path("Producción", "Publicaciones")
for i, p in enumerate(as_list(publicaciones), 1):
    autores = p.get("authors", "")
    fecha = format_date(p.get("date", ""))
    titulo = p.get("title", "")
    revista = p.get("journal", "")
    detalles = "; ".join(list_to_text(p.get("description")))

    print(f"{i}. {autores} ({fecha}). {titulo}. {revista}.")
    if detalles:
        print(f"   Detalles: {detalles}")

print()
print("OTROS ANTECEDENTES RELACIONADOS CON LA ESPECIALIDAD (INVESTIGACIÓN):")
investigacion = get_path("Experiencia", "Investigación")
for i, item in enumerate(as_list(investigacion), 1):
    nombre = item.get("name", "")
    fecha = format_date(item.get("date", ""))
    lugar = item.get("location", "")
    desc = "; ".join(list_to_text(item.get("description")))
    print(f"{i}. {nombre}. {lugar}. Período: {fecha}.")
    if desc:
        print(f"   Detalles: {desc}")
    print()

## d. CURSOS, CONFERENCIAS Y TRABAJOS DE INVESTIGACIÓN

In [ ]:
print_title("d) CURSOS DE ESPECIALIZACIÓN, CONFERENCIAS Y TRABAJOS")

cursos = get_path("Cursos y Congresos")
for i, c in enumerate(as_list(cursos), 1):
    nombre = c.get("name", "")
    fecha = format_date(c.get("date", ""))
    lugar = c.get("location", "")
    duracion = c.get("extension", "")
    idioma = c.get("language", "")
    desc = "; ".join(list_to_text(c.get("description")))

    print(f"{i}. {nombre}.")
    print(f"   Lapso: {fecha}")
    print(f"   Lugar: {lugar}")
    if duracion:
        print(f"   Duración/carga horaria: {duracion}")
    if idioma:
        print(f"   Idioma: {idioma}")
    if desc:
        print(f"   Actividad: {desc}")
    print()

print("TRABAJOS DE INVESTIGACIÓN (EDITOS / PREPRINTS):")
for i, p in enumerate(as_list(get_path("Producción", "Publicaciones")), 1):
    journal = str(p.get("journal", "")).lower()
    if "biorxiv" in journal or "arxiv" in journal:
        print(f"- {p.get('title','')} ({p.get('date','')}) - {p.get('journal','')}")

## e. PARTICIPACIÓN EN CONGRESOS O ACONTECIMIENTOS SIMILARES

In [ ]:
print_title("e) PARTICIPACIÓN EN CONGRESOS O ACONTECIMIENTOS SIMILARES")

posters = get_path("Producción", "Posters y Presentaciones Orales")
for i, item in enumerate(as_list(posters), 1):
    titulo = item.get("title", "")
    evento = item.get("event", "")
    lugar = item.get("location", "")
    fecha = format_date(item.get("date", ""))
    autores = item.get("authors", "")
    calidad = "; ".join(list_to_text(item.get("description")))

    print(f"{i}. {evento}.")
    print(f"   Título/actividad: {titulo}")
    print(f"   Lugar: {lugar}")
    print(f"   Lapso: {fecha}")
    if autores:
        print(f"   Autores/representación: {autores}")
    if calidad:
        print(f"   Calidad de participación: {calidad}")
    print()

## f. ACTUACIÓN EN INSTITUCIONES Y CARGOS EN SECTOR PÚBLICO/PRIVADO

In [ ]:
print_title("f.1) ACTUACIÓN EN UNIVERSIDADES E INSTITUTOS")

secciones_universitarias = ["Académico", "Investigación", "Docencia y Formación", "Proyectos"]
idx = 1
for sec in secciones_universitarias:
    for item in as_list(get_path("Experiencia", sec)):
        nombre = item.get("name", "")
        fecha = format_date(item.get("date", ""))
        lugar = item.get("location", "")
        desc = "; ".join(list_to_text(item.get("description")))
        print(f"{idx}. {nombre}.")
        print(f"   Organismo/entidad: {lugar}")
        print(f"   Lapso: {fecha}")
        if desc:
            print(f"   Tareas: {desc}")
        print()
        idx += 1

print_title("f.2) CARGOS EN ADMINISTRACIÓN PÚBLICA O ACTIVIDAD PRIVADA")
for i, item in enumerate(as_list(get_path("Experiencia", "Profesionales")), 1):
    nombre = item.get("name", "")
    fecha = format_date(item.get("date", ""))
    lugar = item.get("location", "")
    desc = "; ".join(list_to_text(item.get("description")))
    print(f"{i}. {nombre}.")
    print(f"   Entidad/lugar: {lugar}")
    print(f"   Lapso: {fecha}")
    if desc:
        print(f"   Funciones: {desc}")
    print()

## g. FORMACIÓN DE RECURSOS HUMANOS

In [ ]:
print_title("g) FORMACIÓN DE RECURSOS HUMANOS")

docencia = as_list(get_path("Experiencia", "Docencia y Formación"))
claves_formacion = ("supervisión", "supervision", "mentor", "formación", "formacion", "taller")

formacion_rrhh = []
for item in docencia:
    nombre = str(item.get("name", "")).lower()
    texto = " ".join(list_to_text(item.get("description"))).lower()
    if any(c in nombre for c in claves_formacion) or any(c in texto for c in claves_formacion):
        formacion_rrhh.append(item)

print("FORMACIÓN/SUPERVISIÓN DE PERSONAS:")
for i, item in enumerate(formacion_rrhh, 1):
    print(f"{i}. {item.get('name','')}")
    print(f"   Institución/lugar: {item.get('location','')}")
    print(f"   Lapso: {item.get('date','')}")
    for d in list_to_text(item.get('description')):
        print(f"   - {d}")
    print()

print("BECAS Y FINANCIAMIENTOS ACREDITADOS (relacionados):")
investigacion = as_list(get_path("Experiencia", "Investigación"))
claves_becas = ("beca", "funded", "subsidiado", "seal of excellence", "sello de excelencia")
idx = 1
for item in investigacion:
    detalles = list_to_text(item.get("description"))
    hit = [d for d in detalles if any(k in d.lower() for k in claves_becas)]
    if hit:
        print(f"{idx}. {item.get('name','')} ({item.get('date','')})")
        print(f"   Institución/lugar: {item.get('location','')}")
        for h in hit:
            print(f"   - {h}")
        print()
        idx += 1

## h. SÍNTESIS DE APORTES ORIGINALES

In [ ]:
print_title("h) SÍNTESIS DE APORTES ORIGINALES (BORRADOR EDITABLE)")

pubs = as_list(get_path("Producción", "Publicaciones"))
total_pubs = len(pubs)
recent_pubs = [p for p in pubs if str(p.get("date", "")).isdigit() and int(p.get("date")) >= 2023]

print("Texto sugerido (ajustar libremente):")
print()
print("Durante los últimos años he realizado aportes originales en biofísica, microscopía y análisis cuantitativo de bioimágenes, integrando desarrollo metodológico, modelado y aplicaciones biomédicas.")
print(f"La producción científica incluye {total_pubs} publicaciones registradas en el CV, con {len(recent_pubs)} contribuciones recientes (2023 en adelante).")
print("En particular, contribuí al desarrollo y adopción de metodologías reproducibles para análisis de imágenes biológicas, incluyendo proyectos colaborativos internacionales en infraestructura de bioinformática e imagen.")
print("Estos aportes se desarrollaron en instituciones de Argentina, Suecia y colaboraciones europeas, en el lapso 2016-actualidad.")

## i. SÍNTESIS DE ACTUACIÓN PROFESIONAL Y/O EXTENSIÓN UNIVERSITARIA

In [ ]:
print_title("i) SÍNTESIS DE ACTUACIÓN PROFESIONAL Y/O EXTENSIÓN")

prof = as_list(get_path("Experiencia", "Profesionales"))
divulg = as_list(get_path("Producción", "Divulgación Científica"))
doc = as_list(get_path("Experiencia", "Docencia y Formación"))

print("RESUMEN SUGERIDO:")
print()
print("Mi actuación profesional reciente se centró en consultoría y desarrollo de algoritmos de análisis de imágenes para instituciones académicas y privadas, en Argentina y Suecia.")
print("En extensión universitaria y transferencia, participé en actividades de divulgación científica, talleres, cursos intensivos y eventos de formación en análisis de bioimágenes e inteligencia artificial aplicada.")
print()
print("SOPORTE (ítems para copiar):")
print()
print("Actividad profesional:")
for i, p in enumerate(prof, 1):
    print(f"{i}. {p.get('name','')} - {p.get('location','')} ({p.get('date','')})")

print()
print("Extensión/divulgación científica:")
for i, d in enumerate(divulg, 1):
    print(f"{i}. {d.get('title','')} - {d.get('event','')} ({d.get('date','')})")

print()
print("Docencia y formación con impacto de extensión:")
for i, d in enumerate(doc, 1):
    name = str(d.get("name", ""))
    if any(k in name.lower() for k in ["organizador", "workshop", "taller", "khipu", "jornada"]):
        print(f"{i}. {d.get('name','')} - {d.get('location','')} ({d.get('date','')})")

## j. OTROS ELEMENTOS DE JUICIO VALIOSOS

In [ ]:
print_title("j) OTROS ELEMENTOS DE JUICIO VALIOSOS")

print("ELEMENTOS SUGERIDOS (no exhaustivo):")

invest = as_list(get_path("Experiencia", "Investigación"))
destacados = []
for item in invest:
    for d in list_to_text(item.get("description")):
        low = d.lower()
        if any(k in low for k in ["sello de excelencia", "seal of excellence", "beca", "funded", "subsidiado"]):
            destacados.append((item.get("name", ""), item.get("date", ""), d))

for i, (nombre, fecha, detalle) in enumerate(destacados, 1):
    print(f"{i}. {nombre} ({fecha}): {detalle}")

print()
print("Participación internacional (lugares detectados en experiencia):")
lugares = set()
for sec in ["Académico", "Investigación", "Docencia y Formación", "Profesionales"]:
    for item in as_list(get_path("Experiencia", sec)):
        loc = item.get("location", "")
        if loc:
            lugares.add(loc)
for i, loc in enumerate(sorted(lugares), 1):
    print(f"{i}. {loc}")

## k. PLAN DE LABOR DOCENTE, INVESTIGACIÓN Y EXTENSIÓN

In [ ]:
print_title("k) PLAN DE LABOR (BORRADOR BASE)")

plan = """
PROPUESTA GENERAL (editable)

1) Docencia
- Desarrollar una enseñanza basada en problemas reales de análisis de bioimágenes, combinando fundamentos físicos, estadísticos y computacionales.
- Integrar teoría y práctica con actividades de laboratorio reproducibles y uso de herramientas abiertas.
- Fortalecer competencias en adquisición, procesamiento y análisis cuantitativo de imágenes para estudiantes de grado y posgrado.

2) Investigación científica y tecnológica
- Consolidar líneas de trabajo en microscopía cuantitativa, análisis computacional e IA aplicada a imágenes biológicas y médicas.
- Promover proyectos interdisciplinarios y colaboraciones nacionales e internacionales.
- Desarrollar pipelines reproducibles y transferibles a la comunidad académica.

3) Extensión universitaria y transferencia
- Organizar talleres y jornadas de formación para ampliar capacidades en análisis de bioimágenes.
- Vincular docencia e investigación con necesidades de laboratorios, hospitales e instituciones del medio socio-productivo.

4) Actualización permanente
- Incorporar contenidos emergentes (IA, visión por computadora, buenas prácticas de ciencia abierta).
- Revisar periódicamente el programa de la asignatura con base en evidencia pedagógica y avances disciplinares.

NOTA: adaptar este texto al cargo específico (Titular/Asociado/Adjunto) y a la asignatura concreta del concurso.
"""

print(plan)

## l. SOLO RENOVACIÓN: INFORME DE CUMPLIMIENTO DEL PLAN ANTERIOR

In [ ]:
print_title("l) INFORME PARA CONCURSO DE RENOVACIÓN (GUÍA)")

print("Usar solo si el concurso es de renovación.")
print()
print("CHECKLIST DE CONTENIDO:")
print("1. Objetivos del plan anterior")
print("2. Actividades docentes realizadas")
print("3. Actividades de investigación realizadas")
print("4. Actividades de extensión realizadas")
print("5. Resultados verificables y certificaciones adjuntas")
print()
print("CARGOS/ACTIVIDADES VIGENTES DETECTADAS EN CV (base para informe):")

idx = 1
for sec in ["Académico", "Investigación", "Docencia y Formación", "Profesionales"]:
    for item in as_list(get_path("Experiencia", sec)):
        d = item.get("date", "")
        if is_ongoing(d):
            print(f"{idx}. [{sec}] {item.get('name','')} - {item.get('location','')} ({d})")
            idx += 1

if idx == 1:
    print("No se detectaron actividades con 'actual' en la fecha.")